# KADMON — Visualisation locale 3D d’un ou plusieurs neurotracts

**Kantorovich Anatomical Deviation Mapping of Neurotracts**

1. Déposez deux bundles dans le dossier `bundles/`, situé à côté de ce notebook.
2. Modifiez uniquement la cellule **Choix manuels** ci-dessous.
3. Exécutez toutes les cellules dans l'ordre.

Le notebook exécute la configuration choisie et anime dans FURY le déplacement entre la source et sa projection OT.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
BUNDLES_DIR = PROJECT_ROOT / "notebooks" / "bundles"


## Choix manuels

C'est la seule cellule à modifier. Les paramètres scientifiques optimisés sont fixés dans la cellule suivante.

In [2]:
# Paire utilisée lorsque SHOW_ALL_BUNDLES = False.
SOURCE_PATH = BUNDLES_DIR / "103818/nn_8mm/tractosearch_nn_8_0mm_all_CC_1_m_12mpts_rasmm.npy"
TARGET_PATH = BUNDLES_DIR / "433839/nn_8mm/tractosearch_nn_8_0mm_all_CC_1_m_12mpts_rasmm.npy"

# Choisir : quickbundles_partial, quickbundles_sinkhorn,
#           kmeans_partial, kmeans_sinkhorn,
#           binning_partial ou binning_sinkhorn.
DETAILED_CONFIGURATION = "quickbundles_partial"

SHOW_ALL_BUNDLES = False       # True: tous les bundles; False: seulement la paire ci-dessus.
MINIMUM_VISIBLE_DISPLACEMENT_MM = 0.0  # Masquer en noir les déplacements sous ce seuil.
ANIMATION_CYCLE_SECONDS = 4.0  # Durée d'un aller-retour complet.
ANIMATION_FPS = 30             # Fluidité de l'animation.
AMPLITUDE_COLORMAP = "magma"  # Palette continue du déplacement en millimètres.
COLOR_UPPER_PERCENTILE = 95    # P95 évite qu'un extrême écrase l'échelle.


In [3]:
import pandas as pd

from kadmon.comparison import ComparisonConfiguration, compare_configurations
from kadmon.io import load_bundle
from kadmon.visualization import animate_local_displacements_3d

# Paramètres scientifiques optimisés — ne pas modifier pour reproduire les résultats.
from kadmon.defaults import (
    BINNING_PARTIAL_BIN_SIZE,
    BINNING_PARTIAL_BINNING_NB,
    BINNING_PARTIAL_MASS,
    BINNING_PARTIAL_REPRESENTATIVE,
    BINNING_SINKHORN_BIN_SIZE,
    BINNING_SINKHORN_BINNING_NB,
    BINNING_SINKHORN_EPSILON,
    BINNING_SINKHORN_REPRESENTATIVE,
    KMEANS_PARTIAL_K,
    KMEANS_PARTIAL_MASS,
    KMEANS_SINKHORN_EPSILON,
    KMEANS_SINKHORN_K,
    QUICKBUNDLES_PARTIAL_MASS,
    QUICKBUNDLES_PARTIAL_THRESHOLD,
    QUICKBUNDLES_SINKHORN_EPSILON,
    QUICKBUNDLES_SINKHORN_THRESHOLD,
)

N_POINTS = 12
SEED = 42
CONFIGURATION_TITLES = {
    "quickbundles_partial": "QuickBundles + Partial OT",
    "quickbundles_sinkhorn": "QuickBundles + Sinkhorn",
    "kmeans_partial": "K-means + Partial OT",
    "kmeans_sinkhorn": "K-means + Sinkhorn",
    "binning_partial": "Binning + Partial OT",
    "binning_sinkhorn": "Binning + Sinkhorn",
}
if DETAILED_CONFIGURATION not in CONFIGURATION_TITLES:
    raise ValueError(f"Configuration inconnue : {DETAILED_CONFIGURATION!r}")
DETAILED_TITLE = CONFIGURATION_TITLES[DETAILED_CONFIGURATION]
ALL_BUNDLES_DIR = SOURCE_PATH.parent
ALL_BUNDLES_GLOB = f"*_{N_POINTS}mpts_rasmm.npy"


Info: some functions in tractosearch.resampling are faster when 'numba' is installed


## Validation et chargement

In [4]:
if not SOURCE_PATH.is_file():
    raise FileNotFoundError(f"Bundle source introuvable : {SOURCE_PATH}")
if not TARGET_PATH.is_file():
    raise FileNotFoundError(f"Bundle cible introuvable : {TARGET_PATH}")

bundle_a = load_bundle(SOURCE_PATH, n_points=N_POINTS)
bundle_b = load_bundle(TARGET_PATH, n_points=N_POINTS)

print(f"Source : {len(bundle_a)} streamlines, forme={bundle_a.shape}")
print(f"Cible  : {len(bundle_b)} streamlines, forme={bundle_b.shape}")

Source : 10954 streamlines, forme=(10954, 12, 3)
Cible  : 11497 streamlines, forme=(11497, 12, 3)


## Comparaison

In [5]:
def compare_pair(source, target):
    sinkhorn = {"max_iter": 2000, "stop_threshold": 1e-6, "reject_threshold": 1e-5}
    kmeans = {"max_iter": 300, "tol": 1e-4, "seed": SEED}
    binning = {"binning_nb": BINNING_PARTIAL_BINNING_NB, "method": BINNING_PARTIAL_REPRESENTATIVE, "n_points": source.shape[1]}
    configurations = {
        "quickbundles_partial": ComparisonConfiguration("quickbundles", "partial", {"threshold": QUICKBUNDLES_PARTIAL_THRESHOLD}, {"mass": QUICKBUNDLES_PARTIAL_MASS}),
        "quickbundles_sinkhorn": ComparisonConfiguration("quickbundles", "sinkhorn", {"threshold": QUICKBUNDLES_SINKHORN_THRESHOLD}, {**sinkhorn, "epsilon": QUICKBUNDLES_SINKHORN_EPSILON}),
        "kmeans_partial": ComparisonConfiguration("kmeans", "partial", {**kmeans, "n_clusters": KMEANS_PARTIAL_K}, {"mass": KMEANS_PARTIAL_MASS}),
        "kmeans_sinkhorn": ComparisonConfiguration("kmeans", "sinkhorn", {**kmeans, "n_clusters": KMEANS_SINKHORN_K}, {**sinkhorn, "epsilon": KMEANS_SINKHORN_EPSILON}),
        "binning_partial": ComparisonConfiguration("binning", "partial", {**binning, "bin_size": BINNING_PARTIAL_BIN_SIZE}, {"mass": BINNING_PARTIAL_MASS}),
        "binning_sinkhorn": ComparisonConfiguration("binning", "sinkhorn", {**binning, "bin_size": BINNING_SINKHORN_BIN_SIZE, "binning_nb": BINNING_SINKHORN_BINNING_NB, "method": BINNING_SINKHORN_REPRESENTATIVE}, {**sinkhorn, "epsilon": BINNING_SINKHORN_EPSILON}),
    }
    return compare_configurations(source, target, {DETAILED_CONFIGURATION: configurations[DETAILED_CONFIGURATION]})[DETAILED_CONFIGURATION]

result = compare_pair(bundle_a, bundle_b)

## Animation 3D des déplacements

La fenêtre FURY anime un aller-retour entre les représentants source et leur projection barycentrique OT. Le mouvement montre la direction; la couleur indique l’amplitude du déplacement en millimètres selon la légende. Clic-glisser pour tourner, molette pour zoomer.

L’interpolation est une représentation du vecteur OT, pas une trajectoire anatomique ni une déformation physique. Le slider au bas de la fenêtre permet de parcourir manuellement le déplacement de 0 % (source) à 100 % (projection); le bouton **PLAY/PAUSE** contrôle l’animation.

In [6]:
if SHOW_ALL_BUNDLES:
    source_files = sorted(ALL_BUNDLES_DIR.glob(ALL_BUNDLES_GLOB))
    if not source_files:
        raise FileNotFoundError(
            f"Aucun fichier correspondant à {ALL_BUNDLES_GLOB!r} dans {ALL_BUNDLES_DIR}"
        )
    all_bundle_results = {}
    missing_targets = []
    for source_file in source_files:
        target_file = TARGET_PATH.parent / source_file.name
        if not target_file.is_file():
            missing_targets.append(target_file)
            continue
        source_bundle = load_bundle(source_file, n_points=N_POINTS)
        target_bundle = load_bundle(target_file, n_points=N_POINTS)
        all_bundle_results[source_file.stem] = compare_pair(source_bundle, target_bundle)
    if missing_targets:
        print(f"Cibles absentes ignorées : {len(missing_targets)}")
    animation_results = all_bundle_results
else:
    animation_results = result

scene, animation_manager = animate_local_displacements_3d(
    animation_results,
    cycle_duration_s=ANIMATION_CYCLE_SECONDS,
    frames_per_second=ANIMATION_FPS,
    minimum_visible_displacement_mm=MINIMUM_VISIBLE_DISPLACEMENT_MM,
    amplitude_colormap=AMPLITUDE_COLORMAP,
    color_upper_percentile=COLOR_UPPER_PERCENTILE,
)


Animation OT : 0 % = source; 100 % = projection barycentrique.
La couleur code l'amplitude 0–15.54 mm (P95); la direction est portée par l'animation.
Le bouton PLAY/PAUSE contrôle l'animation; déplacer le slider la met en pause.


## Synthèse quantitative

Les distances sont exprimées en millimètres. Le classement utilise le déplacement moyen pondéré de chaque bundle.

In [7]:
summary_results = all_bundle_results if SHOW_ALL_BUNDLES else {SOURCE_PATH.stem: result}
summary_rows = []
for bundle_name, bundle_result in summary_results.items():
    metrics = bundle_result["metrics"]
    summary_rows.append({
        "bundle": bundle_name,
        "mean_mm": metrics["mean_mm"],
        "median_mm": metrics["median_mm"],
        "p95_mm": metrics["p95_mm"],
        "max_mm": metrics["max_mm"],
        "global_distance_mm": metrics["global_distance_mm"],
        "transported_mass": metrics["transported_mass"],
        "source_representatives": metrics["source_n_representatives"],
        "target_representatives": metrics["target_n_representatives"],
    })

summary = pd.DataFrame(summary_rows).sort_values("mean_mm", ascending=False)
display(summary.style.format({
    "mean_mm": "{:.2f}", "median_mm": "{:.2f}",
    "p95_mm": "{:.2f}", "max_mm": "{:.2f}",
    "global_distance_mm": "{:.2f}", "transported_mass": "{:.2f}",
}))

most_displaced = summary.iloc[0]
print(f"Bundle le plus déplacé : {most_displaced.bundle}")
print(f"  déplacement moyen : {most_displaced.mean_mm:.2f} mm")
print(f"  médiane / P95      : {most_displaced.median_mm:.2f} / {most_displaced.p95_mm:.2f} mm")
print(f"Moyenne des bundles : {summary.mean_mm.mean():.2f} mm")

AttributeError: The '.style' accessor requires jinja2